In [ ]:
%xmode Context
%load_ext autoreload
%autoreload 2

import numpy as np
import matplotlib.pyplot as plt

# help locating the package sgpykit (comment out if you installed it)
import sys
from pathlib import Path
PKG_PARENT = Path().resolve().parent
sys.path.insert(0, str(PKG_PARENT))

# load sgpykit
import sgpykit as sg

# Set up logging with sgpykit convenience functions
sg.set_logger_basic_format()
sg.set_logger_info_level()
# For more verbose output, use: sg.set_logger_debug_level()
# For custom format, use: sg.set_logger_custom_format(format)

# sgpykit

## Differences to Sparse Grids Matlab Kit

- sgpykit index arrays are in a 0-based index format (Matlab uses 1-based indexing), so the first element of an array has index 0.
- Scalars are not arrays. The default array type is float64, but you can specify the dtype and number of dimensions.
- Use placeholders (`_`) for unused return arguments like `something, _, another_thing = some_function()` (Matlab allows a flexible number of output arguments, while sgpykit/Python does not).
- Use named arguments in functions to specify variables, as in `evaluate_on_sparse_grid(f, S=None, Sr=my_Sr)`.
- More information about such differences can be found in [differences.md](../differences.md).

In [ ]:
N = 2
w = 3
idxset = lambda i: sum(i - 1)
sg.multiidx_gen(N=N, rule=idxset, w=w, base=1)

In [ ]:
# for 0-based indices
idxset = lambda i: sum(i)
sg.multiidx_gen(N=N, rule=idxset, w=w, base=0)

# PART 1 - INTRODUCTION

## What Is a Sparse Grid?

A sparse grid is a linear combination of many tensor grids on $\mathbb{R}^N$ (parameter space).
Each of the tensor grids included has "few points." With suitable linear combinations
of such grids, it is possible to achieve good accuracy in quadrature and interpolation,
while reducing the computational cost compared to using a single tensor grid.

Run these commands to build a sparse grid and visualize each component.

In [ ]:
N = 2
w = 3

knots = lambda n: sg.knots_CC(n, -1, 1, 'nonprob')
S, C = sg.create_sparse_grid(N, w, knots, sg.lev2knots_doubling)
S[0]
#len(S)
#S.shape
#S.knots


### Visualisation

- For marker styles, see the [matplotlib API](https://matplotlib.org/stable/api/markers_api.html).

In [ ]:
# plot the grid itself
fig, axs = sg.figure_create()
sg.plot_sparse_grid(axs, S, [], 'color', 'k', 'marker', 'o', 'MarkerFaceColor', 'k')
axs.set_title("Knots")

In [ ]:
# each component
fig, axs = sg.figure_create(nrows=2, ncols=4, figsize=(16, 8))
s_max = len(S)
k = 0
for s in range(s_max):
    if len(S[s]) > 0:
        sg.plot_sparse_grid(axs.flat[k], S[s], [], 'color', 'k', 'marker', 'o', 'MarkerFaceColor', 'k')
        k = k + 1
fig.tight_layout()  # auto spacing subplots

### 3D Plot

In [ ]:
N = 3
w = 3
knots = lambda n: sg.knots_CC(n, -1, 1, 'nonprob')
S3, C = sg.create_sparse_grid(N, w, knots, sg.lev2knots_doubling)

In [ ]:
#%matplotlib ipympl
# switch to ipympl matplotlib GUI backend for Jupyter, to be able to rotate 3D view (a bit laggy though)

sg.plot3_sparse_grid(S3, [], 'color', 'k', 'marker', 'o', 'MarkerFaceColor', 'k')

## Tensor Grid

In [ ]:
knots = lambda n: sg.knots_CC(n, -1, 1, 'nonprob')
lev2knots = sg.lev2knots_doubling
#lev2knots = sg.lev2knots_lin
N = 3
ii = np.array([1, 2, 3])
m = sg.lev2knots_doubling(ii)
s = sg.tensor_grid(N, m, knots)
s

## Ingredients of a Sparse Grid

### 1D Knots

Each tensor grid in the sparse grid is built by taking Cartesian products of 1D point distributions.
In general, a different number of points is used in each direction. sgpykit provides
several knot families. These functions also return the quadrature weights associated with the knots
(more on this later).

**Gauss-Legendre points**: quadrature points to approximate integrals of the form
$$
\int_a^b f(x) \frac{1}{b-a} \, dx
$$
with $n$ points.

In [ ]:
# switch back to inline matplotlib GUI backend for Jupyter
%matplotlib inline
sg.clear()  # clear previous plots in cache, otherwise they would appear again in this cell

n = 5
a = 1
b = 4
x, w = sg.knots_uniform(n, a, b)

fig, axs = sg.figure_create(figsize=(10, 8))
sg.plot(axs, x, 0 * x, 'ok', 'MarkerFaceColor', 'k', 'DisplayName', '5 GL points')

**Clenshaw-Curtis points**: nested quadrature points to approximate integrals of the form
$$
\int_a^b f(x) \frac{1}{b-a} \, dx
$$
with $n$ points. If one "doubles" the number of points, the new points will include the old ones.

In [ ]:
n = 5
a = 1
b = 4
x, w = sg.knots_CC(n, a, b)

#fig, axs = figure_create()
sg.plot(axs, x, 1 + 0 * x, 'or', 'MarkerFaceColor', 'r', 'DisplayName', '5 CC points')
fig

In [ ]:
fig, axs = sg.figure_create()
n = 9
a = 1
b = 4
x, _ = sg.knots_uniform(n, a, b)
sg.plot(axs, x, -1 + 0 * x, 'ob', 'MarkerFaceColor', 'b', 'DisplayName', '9 GL points (does NOT includes the 5 points)')

n = 9
a = 1
b = 4
x, _ = sg.knots_CC(n, a, b)
sg.plot(axs, x, 2 + 0 * x, 'og', 'MarkerFaceColor', 'g', 'DisplayName', '9 CC points (includes the 5 points)')

axs.set_ylim([-1.5, 4])
plt.legend()

**Leja points**: nested quadrature points to approximate integrals of the form
$$
\int_a^b f(x) \frac{1}{b-a} \, dx
$$
with $n$ points. Three different kinds of Leja points are available: Line Leja, sym-Line Leja, and p-disk Leja (see leja_points.m for more details). All Leja points are nested by construction.

In [ ]:
n = 5
a = 1
b = 4
x, _ = sg.knots_leja(n, a, b, 'line')

fig, axs = sg.figure_create()
sg.plot(axs, x, 1 + 0 * x, 'or', 'MarkerFaceColor', 'r', 'DisplayName', '5 CC points')

n = 9
a = 1
b = 4
x, _ = sg.knots_leja(n, a, b, 'line')

sg.plot(axs, x, 2 + 0 * x, 'og', 'MarkerFaceColor', 'g', 'DisplayName', '9 Line Leja points')
plt.legend()

In [ ]:
# ------- sym leja ----------
n = 5
a = 1
b = 4
x, _ = sg.knots_leja(n, a, b, 'sym_line')

fig, axs = sg.figure_create()
sg.plot(axs, x, 3 + 0 * x, 'ok', 'MarkerFaceColor', 'k', 'DisplayName', '5 sym-line Leja points')

n = 9
a = 1
b = 4
x, _ = sg.knots_leja(n, a, b, 'sym_line')
sg.plot(axs, x, 4 + 0 * x, 'og', 'MarkerFaceColor', 'g', 'DisplayName', '9 sym-Line Leja points')
plt.legend()

In [ ]:
# ------- p-disk leja ----------
n = 5
a = 1
b = 4
x, _ = sg.knots_leja(n, a, b, 'p_disk')

fig, axs = sg.figure_create()
sg.plot(axs, x, 5 + 0 * x, 'ob', 'MarkerFaceColor', 'b', 'DisplayName', '5 p-Disk Leja points')

n = 9
a = 1
b = 4
x, _ = sg.knots_leja(n, a, b, 'p_disk')
sg.plot(axs, x, 6 + 0 * x, 'og', 'MarkerFaceColor', 'g', 'DisplayName', '9 p-Disk Leja points')

axs.set_ylim([-1.5, 12])
plt.legend()

We also provide equispaced (trapezoidal quadrature rule) and midpoint points,
which should of course be used sparingly as they are not suitable for high-order
interpolation (Runge phenomenon).

In [ ]:
fig, axs = sg.figure_create()  # TODO: \/ changed point symbols

n = 5
a = 1
b = 4
x, _ = sg.knots_midpoint(n, a, b)
sg.plot(axs, x, 1 + 0 * x, 'or', 'MarkerFaceColor', 'r', 'DisplayName', '5 midpoints')

n = 6
a = 1
b = 4
x, _ = sg.knots_trap(n, a, b)
sg.plot(axs, x, 1 + 0 * x, 'ob', 'MarkerFaceColor', 'b', 'DisplayName', '6 trapezoidal points')


n = 10
a = 1
b = 4
x, _ = sg.knots_midpoint(n, a, b)
sg.plot(axs, x, 2 + 0 * x, 'xr', 'MarkerFaceColor', 'r', 'DisplayName', '10 midpoints')

n = 11
a = 1
b = 4
x, _ = sg.knots_trap(n, a, b)
sg.plot(axs, x, 2 + 0 * x, 'xb', 'MarkerFaceColor', 'b', 'DisplayName', '11 trapezoidal points (nested with 6, 6+5=11)')

n = 30
a = 1
b = 4
x, _ = sg.knots_midpoint(n, a, b)
sg.plot(axs, x, 3 + 0 * x, '.r', 'MarkerFaceColor', 'r', 'DisplayName', '30 midpoints (nested with 10 midpoints)')


axs.set_ylim([-0.5, 4.5])
plt.legend()

Quadrature points to approximate integrals with normal PDF, like
$$
\frac{1}{\sqrt{2 \sigma^2 \pi}} \int_\mathbb{R} f(x) \exp\left( -\frac{(x-\mu)^2}{2 \sigma^2} \right) \, dx
$$

In [ ]:
# Gauss-Hermite points
n = 9
mu = 0
sig = 1
x, _ = sg.knots_normal(n, mu, sig)

fig, axs = sg.figure_create()
sg.plot(axs, x, 0 * x, 'ok', 'MarkerFaceColor', 'k', 'DisplayName', '9 GH points')

# Genz-Keister / Kronrod - Patterson Nodes : nested quadrature points to approximate integrals as the previous

n = 3
x, _ = sg.knots_GK(n, 0, 1)
sg.plot(axs, x, 1 + 0 * x, 'or', 'MarkerFaceColor', 'r', 'DisplayName', '3 GK points')

n = 9
x, _ = sg.knots_GK(n, 0, 1)
sg.plot(axs, x, 2 + 0 * x, 'ob', 'MarkerFaceColor', 'b', 'DisplayName', '9 GK points')

plt.legend()

In [ ]:
# Normal-Leja : nested quadrature points to approximate integrals as the previous
fig, axs = sg.figure_create()
mu = 0
sig = 1
n = 3
x, _ = sg.knots_normal_leja(n, mu, sig, 'line')  # another option here is 'sym_line'
sg.plot(axs, x, 3 + 0 * x, 'xr', 'LineWidth', 2, 'MarkerFaceColor', 'r', 'DisplayName', '3 Normal-Leja points')

n = 9
x, _ = sg.knots_normal_leja(n, mu, sig, 'line')
sg.plot(axs, x, 4 + 0 * x, 'xb', 'LineWidth', 2, 'MarkerFaceColor', 'b', 'DisplayName', '9 Normal-Leja points')

axs.set_ylim([-1.5, 7])
plt.legend()

Quadrature points to approximate integrals with exponential PDF, like
$$
\int_0^{\infty} f(x) \lambda e^{ -\lambda x } \, dx
$$

In [ ]:
# Gauss-Laguerre points
n = 12
lmbda = 1
x, _ = sg.knots_exponential(n, lmbda)

fig, axs = sg.figure_create()
sg.plot(axs, x, 0 * x, 'ok', 'MarkerFaceColor', 'k', 'DisplayName', '12 Gauss-Laguerre points')

# and their Leja Counter part

x, _ = sg.knots_exponential_leja(n, lmbda)
sg.plot(axs, x, 1 + 0 * x, 'or', 'LineWidth', 2, 'MarkerFaceColor', 'r', 'DisplayName', '12 Exponential-Leja points')

axs.grid(True)
axs.set_ylim([-0.5, 1.5])
plt.legend()

Quadrature points to approximate integrals with gamma PDF, like
$$
\int_0^\infty f(x) \frac{\beta^{\alpha+1}}{\Gamma(\alpha+1)} \cdot x^\alpha \cdot e^{-\beta x} \, dx
$$

In [ ]:
#  Gauss-Generalized Laguerre points:  with n points
n = 12
alpha = 1
beta = 2

x, _ = sg.knots_gamma(n, alpha, beta)

fig, axs = sg.figure_create()
sg.plot(axs, x, 0 * x, 'ok', 'MarkerFaceColor', 'k', 'DisplayName', '12 Generalized Gauss-Laguerre points')

# and their Leja Counter part

x, _ = sg.knots_gamma_leja(n, alpha, beta)
sg.plot(axs, x, 1 + 0 * x, 'or', 'LineWidth', 2, 'MarkerFaceColor', 'r', 'DisplayName', '12 Gamma-Leja points')

axs.grid(True)
axs.set_ylim([-0.5, 1.5])
plt.legend()

Quadrature points to approximate integrals with beta PDF, like

$$
\int_{x_a}^{x_b} f(x) \frac{\Gamma(\alpha+\beta+2)}{\Gamma(\alpha+1) \cdot \Gamma(\beta+1) \cdot (x_b-x_a)^{\alpha+\beta+1}} \cdot (x-x_a)^\alpha \cdot (x_b-x)^\beta \, dx
$$


In [ ]:
#  Gauss-Jacobi points: with n points
n = 12
x_a = 1
x_b = 3
alpha = -0.5
beta = 0.5

x, _ = sg.knots_beta(n, alpha, beta, x_a, x_b)

fig, axs = sg.figure_create()
sg.plot(axs, x, 0 * x, 'ok', 'MarkerFaceColor', 'k', 'DisplayName', '12 Gauss-Jacobi points')

# and their Leja Counter part

n = 12
x, _ = sg.knots_beta_leja(n, alpha, beta, x_a, x_b, 'line')
sg.plot(axs, x, 1 + 0 * x, 'or', 'LineWidth', 2, 'MarkerFaceColor', 'r', 'DisplayName', '12 Beta-Leja points')

n = 12
x, _ = sg.knots_beta_leja(n, alpha, beta, x_a, x_b, 'sym_line')  # recommended only if alpha = beta
sg.plot(axs, x, 2 + 0 * x, 'ob', 'LineWidth', 2, 'MarkerFaceColor', 'b', 'DisplayName', '12 Sym. Beta-Leja points')

axs.grid(True)
axs.set_ylim([-0.5, 3])
plt.legend()

# WARNINGS: depending on the parameters invalid values can occur in the output leading to lesser points than prompted, and warning messages occur

**Triangular-Leja points**: quadrature points to approximate integrals with triangular PDF (i.e., a linear decreasing PDF over the interval $[a,b]$), like
$$
\int_a^b f(x) \cdot \frac{2}{(b-a)^2} \cdot (b-x) \, dx
$$

In [ ]:
n = 12
a = 0
b = 2
x, _ = sg.knots_triangular_leja(n, a, b)

fig, axs = sg.figure_create()
sg.plot(axs, x, 0 * x, 'ok', 'MarkerFaceColor', 'k', 'DisplayName', '12 Triangular-Leja points')

axs.grid(True)
plt.legend()

### lev2knots Functions

In view of building sparse grids, it is useful to order quadrature/interpolation rules in sequences, i.e. to
introduce levels for the rules. sgpykit provides 5 functions to this end:

Adds 1 point from one level to the next: consecutive quadrature/interpolation rules have
1, 2, 3, 4, ... points.

In [ ]:
sg.lev2knots_lin([1, 2, 3, 4, 5])


In [ ]:
# -> lev2knots_2step
#
# adds 2 points from one level to the next: consecutive quadrature/interpolation rules have
# 1,3,5,7,..points

sg.lev2knots_2step([1, 2, 3, 4, 5])

In [ ]:
# -> lev2knots_doubling
#
# "doubles" the number of points from one level to the next: consecutive rules have 1,3,5,9,17... points

sg.lev2knots_doubling([1, 2, 3, 4, 5])

In [ ]:
# -> lev2knots_tripling
#
# triples the number of points from one level to the next: consecutive rules have 1,3,9,27... points.


sg.lev2knots_tripling([1, 2, 3, 4, 5])

In [ ]:
# -> lev2knots_GK
#
# needed when using GK knots which are tabulated. 
# Consecutive rules have 1,3,9,19,35 points. 
# The latter is the finest resolution possible

sg.lev2knots_GK([1, 2, 3, 4, 5])

### Multi-Index Set

The last ingredient to specify when building a sparse grid is the set of tensor grids to be used. The
algorithm will then take care of computing the coefficients of the linear combination of these grids
(note that such coefficients may be 0 as well).

The most convenient way to specify tensor grids is to use multi-index notation: every grid is
associated with a multi-index, that is a vector of integer numbers.
Each number in the vector tells the level of the quadrature rule used in each direction
of the parameter space. For example, the multi-index $[3, 5]$ is associated with the tensor grid
using a quadrature rule of level 3 in direction 1 and level 5 in direction 2. The actual number of points in
each direction depends on the level-knots relation specified by the lev2knots_\*\*\* function.

In [ ]:
N = 2
ii = [3, 5]
knots = lambda n: sg.knots_uniform(n, -1, 1, 'nonprob')

S_lin = sg.tensor_grid(N, sg.lev2knots_lin(ii), knots)
S_doub = sg.tensor_grid(N, sg.lev2knots_doubling(ii), knots)

fig, axs = sg.figure_create()
sg.plot_sparse_grid(axs, S_doub, [], 'color', 'r', 'marker', 's', 'MarkerFaceColor', 'r', 'DisplayName', 'lev2knots-nested')
sg.plot_sparse_grid(axs, S_lin, [], 'color', 'k', 'marker', 'o', 'MarkerFaceColor', 'k', 'DisplayName', 'lev2knots-lin')
plt.legend()

There are two ways of specifying the set of multi-indices to be used.

1) The first one is to use the parameters "level" and "idxset" of the function [`create_sparse_grid()`](https://uncertaintyhub.github.io/sgpykit-doc/_autosummary/sgpykit.main.html#sgpykit.main.create_sparse_grid). In this case, the multi-index set will include all the multi-indices that satisfy the inequality

$$
\text{idxset}(\mathbf{i}) \leq \text{level}
$$

By default, idxset is set to $\text{idxset}(\mathbf{i}) = \sum (i_n - 1)$. The combination of idxset function and lev2knots function
defines the sparse grid type: using $\text{idxset}(\mathbf{i}) = \sum (i_n - 1)$ with lev2knots_lin results in the so-called TD
(Total Degree) tensor grid, while using it with lev2knots_doubling gives the original SM (Smolyak) grid.
Some choices are available by using the function

$$
[\text{lev2nodes}, \text{idxset}] = \text{define\_functions\_for\_rule}(\text{rule}, \text{rates})
$$

but any other set satisfying the so-called "admissibility condition" (see e.g. Gerstner-Griebel "Dimension-Adaptive Tensor-Product Quadrature") can be used.

In [ ]:
N = 2
knots = lambda n: sg.knots_uniform(n, -1, 1, 'nonprob')
w = 5  # level

lev2knots, idxset = sg.define_functions_for_rule('TD', N)
S_TD, _ = sg.create_sparse_grid(N, w, knots, lev2knots, idxset)  # grid

lev2knots, idxset = sg.define_functions_for_rule('HC', N)
S_HC, _ = sg.create_sparse_grid(N, w, knots, lev2knots, idxset)  # grid

# plot the grid itself
fig, axs = sg.figure_create()
sg.plot_sparse_grid(axs, S_TD, [], 'color', 'k', 'marker', 'o', 'MarkerFaceColor', 'k', 'DisplayName', 'TD-grid')
plt.legend()

In [ ]:
fig, axs = sg.figure_create()
sg.plot_sparse_grid(axs, S_HC, [], 'color', 'k', 'marker', 'o', 'MarkerFaceColor', 'k', 'DisplayName', 'HC-grid')
plt.legend()

2) The second one is to use the function [`create_sparse_grid_multiidx_set()`](https://uncertaintyhub.github.io/sgpykit-doc/_autosummary/sgpykit.main.html#sgpykit.main.create_sparse_grid_multiidx_set), where one specifies exactly
the set of multi-indices that one wishes to use. Again, the set has to satisfy
the "admissibility condition," and the rows have to be in lexicographic order.

In [ ]:
np.all(C[:-1] <= C[1:], axis=1)

In [ ]:
C = [
    [0, 0],
    [0, 1],
    [0, 2],
    [0, 3],
    [1, 0],
    [1, 1],
]
adm, C = sg.check_set_admissibility(C)

N = 2
knots = lambda n: sg.knots_uniform(n, -1, 1, 'nonprob')
lev2knots, idxset = sg.define_functions_for_rule('HC', N)

S_M, _ = sg.create_sparse_grid_multiidx_set(C, knots, lev2knots)

fig, axs = sg.figure_create()
sg.plot_sparse_grid(axs, S_M, [], 'color', 'b', 'marker', 'd', 'MarkerFaceColor', 'b', 'DisplayName', 'Multiidx set')
axs.set_xlim([-1, 1])
axs.set_ylim([-1, 1])
plt.legend()

The package provides two functions to generate multi-index sets.

a) [`multiidx_box_set()`](https://uncertaintyhub.github.io/sgpykit-doc/_autosummary/sgpykit.tools.idxset_functions.html#sgpykit.tools.idxset_functions.multiidx_box_set) generates all multi-indices $\mathbf{j}$ that are component-wise less than or
equal to some other index $\mathbf{i}$. The minimal value of the components of the indices to be generated can be either 0 or 1. For instance:

In [ ]:
jj = [2, 3]
C, _ = sg.multiidx_box_set(jj, 0)
D, _ = sg.multiidx_box_set(jj, 1)

# the package comes with a convenience function to plot a multiidx set

fig, axs = sg.figure_create()
sg.plot_multiidx_set(axs, C, 'xr', 'MarkerFaceColor', 'r', 'LineWidth', 2, 'MarkerSize', 12, 'DisplayName', 'Multiidx box set, min=0')
sg.plot_multiidx_set(axs, D, 'ok', 'MarkerFaceColor', 'k', 'DisplayName', 'Multiidx box set, min=1')

axs.set_xlim([-0.5, 4])
axs.set_ylim([-0.5, 4])
plt.legend()

b) [`multiidx_gen()`](https://uncertaintyhub.github.io/sgpykit-doc/_autosummary/sgpykit.tools.idxset_functions.html#sgpykit.tools.idxset_functions.multiidx_gen) generates the set of all indices $\mathbf{i}$ such that $\text{rule}(\mathbf{i}) \leq w$, where rule is a function that takes as input a row vector
 (or a matrix where each multi-index is stored as a row) and returns a scalar value (or a column vector with the result of the operation applied
 to each row of the input index vector). Again, the minimum index can be 0 or 1.

In [ ]:
N = 2
w = 7
rule = lambda I: sum(I)
E = sg.multiidx_gen(N, rule, w, 0)
F = sg.multiidx_gen(N, rule, w, 1)

fig, axs = sg.figure_create()
sg.plot_multiidx_set(axs, E, 'xr', 'MarkerFaceColor', 'r', 'LineWidth', 2, 'MarkerSize', 12, 'DisplayName', 'Multiidx gen, min=0')
sg.plot_multiidx_set(axs, F, 'ok', 'MarkerFaceColor', 'k', 'DisplayName', 'Multiidx gen, min=1')

axs.set_xlim([-0.5, 8])
axs.set_ylim([-0.5, 8])
plt.legend()

Incidentally, [`plot_multiidx_set()`](https://uncertaintyhub.github.io/sgpykit-doc/_autosummary/sgpykit.tools.idxset_functions.html#sgpykit.tools.idxset_functions.plot_multiidx_set) works also for $N=3$. For larger dimensions, one needs to input the subset of dimensions that are to be plotted.

In [ ]:
G, _ = sg.multiidx_box_set([2, 3, 5], 1)
fig, axs = sg.figure_create(dims=3)
sg.plot_multiidx_set(axs, G)

In [ ]:
H, _ = sg.multiidx_box_set([2, 3, 5, 4], 1)
fig, axs = sg.figure_create(dims=3)
sg.plot_multiidx_set(axs, H[:, [1, 3]])

When building a large sparse grid, it might be useful to recycle from previous grids to speed up the computation.

In [ ]:
knots = lambda n: sg.knots_normal(n, 0, 1)
lev2knots = sg.lev2knots_lin

N = 40
w = 4
S, _ = sg.create_sparse_grid(N, w, knots, lev2knots, lambda i: np.prod(i + 1))

In [ ]:
sg.set_logger_debug_level()

w = 5

with sg.misc.time_block('build grid without recycling'):
    T, _ = sg.create_sparse_grid(N, w, knots, lev2knots, lambda i: np.prod(i + 1))

with sg.misc.time_block('build grid with recycling'):
    T_rec, _ = sg.create_sparse_grid(N, w, knots, lev2knots, lambda i: np.prod(i + 1), S)

assert T.isequal_to(T_rec), "Results Mismatch"

This is useful in iterative loops like:

In [ ]:
N = 20
w = 4
knots = lambda n: sg.knots_normal(n, 0, 1)
lev2knots = sg.lev2knots_lin

sg.set_logger_info_level()

with sg.misc.time_block():
    for w in range(7):
        # build grid
        T, _ = sg.create_sparse_grid(N, w, knots, lev2knots, lambda i: np.prod(i + 1))
        # then do something ...


with sg.misc.time_block():
    T_old = []
    for w in range(7):
        # build grid
        T, _ = sg.create_sparse_grid(N, w, knots, lev2knots, lambda i: np.prod(i + 1), T_old)
        T_old = T
        # then do something ...


The same functionality is also available for [`create_sparse_grid_multiidx_set()`](https://uncertaintyhub.github.io/sgpykit-doc/_autosummary/sgpykit.main.html#sgpykit.main.create_sparse_grid_multiidx_set).

In [ ]:
N = 5
w = 4
knots = lambda n: sg.knots_normal(n, 0, 1)
lev2knots = sg.lev2knots_lin

ibox = [3, 4, 2, 4, 2]
_, C = sg.multiidx_box_set(ibox, 0)
D, _ = sg.matlab.sortrows(np.vstack((C, [1, 4, 1, 1, 5])))

S, _ = sg.create_sparse_grid_multiidx_set(C, knots, lev2knots)

In [ ]:
sg.set_logger_debug_level()

with sg.misc.time_block():
    T, _ = sg.create_sparse_grid_multiidx_set(D, knots, lev2knots)

with sg.misc.time_block():
    T_rec, _ = sg.create_sparse_grid_multiidx_set(D, knots, lev2knots, S)

assert T.isequal_to(T_rec), "Results Mismatch"

## Data Structures

A sparse grid is represented as a vector of structures. Each element is a tensor grid, with fields
containing the knots, the corresponding integration weights, its coefficient in the linear combination,
and the number of points.

Each tensor grid structure contains the following fields (also see the [SGMK manual online](https://sites.google.com/view/sparse-grids-kit#h.p_IBOqq8I5k39K)):

- **idx**: the 0-based multi-index $\mathbf{i} \in \mathcal{I}\subset\mathbb{N}^N$ corresponding to the current tensor grid
- **knots**: matrix collecting the knots $\mathcal{T}_{\mathbf{i}}$, each knot being a row vector
- **weights**: vector of the quadrature weights $\omega_{m(\mathbf{i})}^{(\mathbf{j})}$ corresponding to the knots
- **size**: size of the tensor grid, i.e. the number of knots $M_{\mathbf{i}} = \prod_{n=0}^{N-1} m(i_n)$
- **knots_per_dim**: list with $N$ components, each component collecting in an array the set of one-dimensional knots $\mathcal{T}_{n, i_n}$ used to build the tensor grid
- **m**: vector collecting the number of knots used in each of the $N$ directions $m(\mathbf{i}) = [m(i_0), m(i_1), \ldots, m(i_{N-1})]$
- **coeff**: the coefficients $c_{\mathbf{i}}$ of the sparse grid in the combination technique formulas

Note that sparse grid knots (and in general knots $\mathbf{y} \in \Gamma$) are always stored in sgpykit as row vectors 
(and sets of knots such as $\mathbf{S}.\text{knots}$ are stored as matrices where knots are rows).

Reduced sparse grids contain the following fields:

- **knots**: matrix collecting the list of non-repeated knots, i.e., the set $\mathcal{T}_{\mathcal{I}}$
- **weights**: vector of quadrature weights corresponding to the knots above
- **size**: size of the sparse grid, i.e. the number of non-repeated knots
- **m**: 0-based index array that maps each knot of $\mathbf{Sr}.\text{knots}$ to their original position in $\mathbf{S}.\text{knots}$ (if they have been retained as unique representative of several repeated knots)
- **n**: 0-based index array that maps each knot of $\mathbf{S}.\text{knots}$ to $\mathbf{Sr}.\text{knots}$

In sgpykit, these grids are stored as customizable Structure Arrays, where additional fields can be added by the user.

### Modify the Domain of a Sparse Grid

It is easy to modify the domain of a sparse grid from $(-1,1)^N$ to other hyper-rectangles.

In [ ]:
N = 2
w = 4
# generate knots on the desired hyper-rectangle (here (0,2)^2 )
knots = lambda n: sg.knots_CC(n, 0, 2, 'nonprob')
S, _ = sg.create_sparse_grid(N, w, knots, sg.lev2knots_doubling)

# plot the grid itself
fig, axs = sg.figure_create()
sg.plot_sparse_grid(axs, S, [], '.k', label='grid S')  # or 'DisplayName','grid S'

plt.legend()

One can mix different intervals and different knot families on different directions.

In [ ]:
N = 2
knots1 = lambda n: sg.knots_CC(n, 0, 2, 'nonprob')
knots2 = lambda n: sg.knots_uniform(n, -1, 5, 'nonprob')
w = 4
S, _ = sg.create_sparse_grid(N, w, [knots1, knots2], [sg.lev2knots_doubling, sg.lev2knots_lin])

In [ ]:
fig, axs = sg.figure_create()
sg.plot_sparse_grid(axs, S, [], '.k', label='grid S2')
plt.legend()

## Reduce a Sparse Grid

The tensor grids forming the sparse grid may have points in common (even when using non-nested points).
To save computational time during, e.g., evaluation of a function on a sparse grid, it is important
to get rid of these repetitions. To this end, use the function [`reduce_sparse_grid()`](https://uncertaintyhub.github.io/sgpykit-doc/_autosummary/sgpykit.main.html#sgpykit.main.reduce_sparse_grid). The quadrature weights
are of course consistently modified. The field "size" tells the number of points in the reduced grid.

In [ ]:
N = 2
w = 1
knots1 = lambda n: sg.knots_CC(n, -1, 1, 'nonprob')


lev2nodes, idxset = sg.define_functions_for_rule('SM', N)
S, _ = sg.create_sparse_grid(N, w, knots, lev2nodes, idxset)
Sr = sg.reduce_sparse_grid(S)

In [ ]:
N = 2
w = 1
knots = lambda n: sg.knots_CC(n, -1, 1, 'nonprob')
S, _ = sg.create_sparse_grid(N, w, knots, sg.lev2knots_doubling)
Sr = sg.reduce_sparse_grid(S)

In [ ]:
S.m

In [ ]:
Sr

In [ ]:
print(f'size of original grid: {np.shape(sg.misc.reshape_nested_lists_to_nrows(S.knots))}')
print(f'size of reduced  grid: {Sr.knots.shape}')
print(f'Sr.size: {Sr.size}')

In [ ]:
fig, axs = sg.figure_create(nrows=1, ncols=2, figsize=(8, 4))
h = sg.plot_sparse_grid(axs[0], S, [], 'color', 'b', 'marker', 'd', 'MarkerFaceColor', 'b', 'DisplayName', 'original grid')
axs[0].legend()
h = sg.plot_sparse_grid(axs[1], Sr, [], 'color', 'b', 'marker', 'd', 'MarkerFaceColor', 'b', 'DisplayName', 'reduced grid')
axs[1].legend()
fig.tight_layout()  # auto spacing subplots

The Kit provides a short-hand to create and reduce a "vanilla sparse grid", i.e.
- Clenshaw-Curtis points in $[-1,1]$
- lev2knots_doubling
- multi-index set: $\sum (ii) \leq w$
(cf [`define_functions_for_rule('SM')`](https://uncertaintyhub.github.io/sgpykit-doc/_autosummary/sgpykit.tools.idxset_functions.html#sgpykit.tools.idxset_functions.define_functions_for_rule) )

In [ ]:
N = 2
w = 3
S, Sr, _ = sg.create_sparse_grid_quick_preset(N, w)

## NEXT: Evaluate Functions on Sparse Grids

=> **[Go to tutorial "Integration"](01-evaluation.ipynb)**